# Laboratorio 2.3 · ELT con dbt sobre DuckDB

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la tienda. En la 2.1 y la 2.2 transformamos bronze en Python (ETL).
Hoy hacemos **lo mismo dentro del motor** (ELT) con dbt, y comprobamos que da exactamente el mismo resultado.

| Parte | Qué haces | La pregunta |
|---|---|---|
| 1 | Bronze sin copiar | ¿Hace falta cargar bronze al warehouse? |
| 2 | El proyecto dbt | ¿Qué es un modelo y qué hace `ref()`? |
| 3 | Las capas | ¿Cómo se ve silver y gold en SQL? |
| 4 | Snapshot | La SCD tipo 2 de la 2.2, en un archivo YAML |
| 5 | ETL contra ELT | ¿Dan lo mismo? |
| 6 | Pruebas | Genéricas, de lote y unitarias |
| 7 | Un lote roto | ¿Se detiene gold? |
| 8 | El tablero | Lo que ve el negocio |

## Preparación (tarda un par de minutos: instala dbt)

In [ ]:
%%capture
!pip -q install -U "dbt-core>=1.10" "dbt-duckdb>=1.9" "polars>=1.10" "pandera[polars]>=0.20" duckdb pyarrow

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(".")
print("Paquete instalado: transformacion/ (Python) y elt/ (dbt)")

In [ ]:
import os, warnings, subprocess
import duckdb, polars as pl
from IPython.display import Markdown, display
from transformacion import generar_bronze
warnings.filterwarnings("ignore")
pl.Config(tbl_rows=16, fmt_str_lengths=50, tbl_width_chars=160)
os.environ["DBT_DUCKDB_PATH"] = "tienda.duckdb"        # relativo a elt/

def consulta(sql):
    # Abre y CIERRA la base en cada consulta: si el notebook la deja abierta, dbt no puede escribir en ella.
    with duckdb.connect("elt/tienda.duckdb", read_only=True) as c:
        return c.sql(sql).pl()

def ver_archivo(ruta, lenguaje="sql"):
    display(Markdown(f"`{ruta}`\n```{lenguaje}\n{open(ruta, encoding='utf-8').read()}\n```"))

print(generar_bronze.generar("landing"))     # las mismas 14 cargas de la 2.1 y la 2.2
!dbt --version 2>&1 | grep -i installed

## Parte 1 · Bronze sin copiar: la E y la L ya ocurrieron

En ELT, la ingesta (sesiones 1.2 y 1.3) **extrae y carga** tal cual; la **transformación** ocurre después,
dentro del motor. Con DuckDB ni siquiera hace falta copiar bronze: el motor lee los Parquet de la landing zone directamente.

In [ ]:
duckdb.sql('''
    SELECT fecha_carga, count(*) AS filas
    FROM read_parquet('landing/bronze/pedidos_web/**/*.parquet', hive_partitioning = true)
    GROUP BY ALL ORDER BY ALL LIMIT 5
''').pl()

En BigQuery esto sería una **tabla externa** sobre Cloud Storage; en AWS, una tabla de Athena sobre S3.
El proyecto dbt declara bronze como **fuente** (`source`) en un archivo YAML:

In [ ]:
ver_archivo("elt/models/staging/_fuentes.yml", "yaml")

## Parte 2 · El proyecto dbt

Un **modelo** es un archivo `.sql` con un `SELECT`. dbt decide si lo crea como vista o tabla, en qué esquema,
y en qué orden correrlo: el orden sale de `ref()` y `source()`, que arman el grafo de dependencias.

In [ ]:
!find elt -type f \( -name "*.sql" -o -name "*.yml" -o -name "*.csv" \) -not -path "*/target/*" | sort

Compara el modelo del punto de venta con la función de Python de la 2.1 (`limpieza.conformar_tienda`): la misma lógica, ahora en SQL.

In [ ]:
ver_archivo("elt/models/staging/stg_pedidos_tienda.sql")

In [ ]:
!cd elt && dbt seed --profiles-dir . --quiet && echo "semillas cargadas"
!cd elt && dbt ls --profiles-dir . --resource-type model --output name --quiet

Los **seeds** son catálogos chicos versionados en el repo (monedas, ciudades, estados del punto de venta).
En la 2.1 eran diccionarios de Python; aquí son tablas que cualquier analista puede leer y corregir.

¿Qué hace `ref()`? dbt lo cambia por el nombre real de la tabla **y** registra la dependencia:

In [ ]:
!cd elt && dbt compile --profiles-dir . --select int_pedidos_validos --quiet
!cd elt && dbt ls --profiles-dir . --select +mart_ventas_diarias --output name --quiet

`+mart_ventas_diarias` significa «el modelo y todo lo que necesita»: es el linaje de la tabla del tablero.

## Parte 3 · Las capas: staging, intermediate y marts

| Carpeta | Capa | Materialización | Qué hace |
|---|---|---|---|
| `staging/` | silver | vista | Limpiar y conformar cada fuente |
| `intermediate/` | silver | tabla | Unir, deduplicar, cuarentena, tipo de cambio «as of» |
| `marts/` | gold | tabla | Dimensiones, hechos y la tabla del tablero |

In [ ]:
!cd elt && dbt build --profiles-dir . --select staging intermediate --exclude stg_clientes_foto --quiet && echo "silver listo"
consulta("SELECT table_schema, table_name, table_type FROM information_schema.tables WHERE table_schema IN ('silver','gold','referencia') ORDER BY ALL")

La deduplicación de la 2.1 (`ultima_version`) en SQL: una función de ventana con el mismo desempate.

In [ ]:
ver_archivo("elt/models/intermediate/int_pedidos_unicos.sql")
consulta("SELECT origen, count(*) AS pedidos FROM silver.int_pedidos_unicos GROUP BY ALL")

Y la cuarentena de la 2.2, como un modelo más: las filas que no cumplen el contrato, con su motivo.

In [ ]:
consulta("SELECT motivo, count(*) AS filas FROM silver.int_pedidos_cuarentena GROUP BY ALL")

El join «as of» de la 2.1 existe en SQL: `ASOF LEFT JOIN`.

In [ ]:
ver_archivo("elt/models/intermediate/int_pedidos_en_pesos.sql")

## Parte 4 · Snapshot: la SCD tipo 2 en un archivo YAML

En la 2.2 escribimos `scd2()` a mano. dbt lo hace con un **snapshot**: cada vez que corre, compara el estado actual
contra la versión vigente; si cambió, la cierra y abre otra.

In [ ]:
ver_archivo("elt/snapshots/snap_clientes.yml", "yaml")

El detalle: un snapshot solo conoce **el estado actual**. En producción corre cada noche sobre la foto del día.
Para reconstruir 14 días, `correr.py` aplica las fotos **una por una, en orden**:

In [ ]:
!python elt/correr.py --landing landing

In [ ]:
consulta("SELECT cliente_id, ciudad, segmento, dbt_valid_from, dbt_valid_to FROM silver.snap_clientes WHERE cliente_id = 42 ORDER BY dbt_valid_from")

Las mismas tres versiones del cliente 42 que en la 2.2. Y la convención de «la primera versión vale desde siempre» vive en gold:

In [ ]:
ver_archivo("elt/models/marts/dim_cliente_historia.sql")

## Parte 5 · ETL contra ELT: ¿dan lo mismo?

Corremos el pipeline de Python de la 2.2 sobre el **mismo bronze** y comparamos.

In [ ]:
from transformacion import pipeline
etl = pipeline.correr("landing", "lakehouse")
py = pl.read_parquet("lakehouse/gold/ventas_diarias.parquet")
sql = consulta("SELECT * FROM gold.mart_ventas_diarias")
pl.DataFrame({
    "medida": ["pedidos en silver", "pedidos en cuarentena", "versiones de clientes", "ingreso total (MXN)"],
    "ETL en Python (2.2)": [etl["silver_pedidos"], etl["cuarentena"], etl["clientes_versiones"], round(py["ingreso_mxn"].sum())],
    "ELT en dbt (hoy)": [consulta("SELECT count(*) FROM gold.fct_pedidos").item(),
                         consulta("SELECT count(*) FROM silver.int_pedidos_cuarentena").item(),
                         consulta("SELECT count(*) FROM gold.dim_cliente_historia").item(),
                         round(sql["ingreso_mxn"].sum())],
})

Mismo resultado, dos herramientas. La pregunta ya no es «¿cuál es correcta?» sino **¿cuál conviene?**
(lámina «ETL o ELT»): quién lo mantiene, dónde viven los datos y cuánto cuesta mover los datos al cómputo.

## Parte 6 · Pruebas

`dbt build` corre cada modelo y, enseguida, sus pruebas. Si una prueba con severidad `error` falla,
dbt **no construye lo que depende de ese modelo**.

In [ ]:
import json
resultados = json.load(open("elt/target/run_results.json"))["results"]
nombre = lambda u: u.split(".")[3] if u.startswith("unit_test") else u.split(".")[2]
pl.DataFrame([{"nodo": r["unique_id"].split(".")[0], "nombre": nombre(r["unique_id"]), "estado": r["status"],
               "filas_que_fallan": r.get("failures")} for r in resultados]).filter(pl.col("nodo").is_in(["test", "unit_test"]))

Tres tipos de prueba:

- **Genéricas** (en YAML): `unique`, `not_null`, `accepted_values`, `relationships` y las nuestras (`llave_unica`, `valor_maximo`).
- **Singulares** (un `.sql` en `tests/`): si devuelve filas, falla. Ahí están las reglas de **lote** de la 2.2.
- **Unitarias** (`unit_tests` en YAML): datos de entrada hechos a mano y el resultado esperado. Como pytest, sin tocar la base.

La de `relationships` tiene severidad `warn`: hay pedidos de clientes que no existen en el CRM. Avisa, no detiene.

In [ ]:
ver_archivo("elt/tests/assert_rechazo_por_carga.sql")
!cd elt && dbt test --profiles-dir . --select test_type:unit --quiet && echo "pruebas unitarias: todas pasan"

## Parte 7 · Un lote roto: ¿se detiene gold?

Generamos la carga rota del 22/09 de la 2.2 (totales en centavos, un estado nuevo, monedas nulas) y corremos solo `dbt build`,
como lo haría el orquestador cada noche.

In [ ]:
antes = consulta("SELECT round(sum(ingreso_mxn))::BIGINT AS ingreso, max(fecha_negocio) AS hasta FROM gold.mart_ventas_diarias")
generar_bronze.generar("landing", lote_malo="ruidoso")
!python elt/correr.py --landing landing --solo-build | tail -3
despues = consulta("SELECT round(sum(ingreso_mxn))::BIGINT AS ingreso, max(fecha_negocio) AS hasta FROM gold.mart_ventas_diarias")
pl.concat([antes.with_columns(momento=pl.lit("antes")), despues.with_columns(momento=pl.lit("después"))])

In [ ]:
resultados = json.load(open("elt/target/run_results.json"))["results"]
pl.DataFrame([{"nombre": nombre(r["unique_id"]), "estado": r["status"]} for r in resultados]).filter(pl.col("estado") != "success").filter(pl.col("estado") != "pass")

Falló `assert_rechazo_por_carga` y dbt **saltó** (`skipped`) todo lo que dependía de silver: gold quedó como estaba.
Es la falla cerrada de la 2.2, sin escribir el código de publicar todo o nada.

Ojo con un detalle: las tablas de silver **sí** se reconstruyeron con el lote roto (el modelo corre antes que su prueba).
Si silver lo consumen otros equipos, se necesita una capa intermedia de staging o el patrón *write-audit-publish*.

Regresamos a las 14 cargas buenas. ¿Basta con volver a correr `dbt build`?

In [ ]:
generar_bronze.generar("landing")
!python elt/correr.py --landing landing --solo-build | tail -1
consulta("SELECT cliente_id, dbt_valid_from, dbt_valid_to FROM silver.snap_clientes WHERE dbt_valid_to < dbt_valid_from LIMIT 3")

No. Durante la corrida rota, el snapshot **sí** capturó la foto del CRM del 22/09 (no depende de los pedidos, así que nadie lo detuvo).
Ahora la foto más reciente vuelve a ser la del 21: aplicarla después de la del 22 crea versiones que terminan antes de empezar,
y la prueba `assert_historia_en_orden` lo detecta.

Un snapshot no se puede deshacer. La salida honesta es la misma de la 2.2: **reconstruir la historia** desde todas las fotos, en orden.

In [ ]:
!python elt/correr.py --landing landing
consulta("SELECT count(*) AS versiones, count(*) FILTER (WHERE dbt_valid_to < dbt_valid_from) AS al_reves FROM silver.snap_clientes")

## Parte 8 · El tablero

El tablero lee **solo gold** y no corrige nada. En el kit local está en `tablero/app.py` (Streamlit):
`streamlit run tablero/app.py`. En Colab, una vista rápida de lo mismo:

In [ ]:
import matplotlib.pyplot as plt
dia = consulta("SELECT fecha_negocio, sum(ingreso_mxn) AS ingreso FROM gold.mart_ventas_diarias WHERE fecha_negocio >= DATE '2026-09-08' GROUP BY 1 ORDER BY 1")
ciudad = consulta("SELECT ciudad, sum(ingreso_mxn) AS ingreso FROM gold.mart_ventas_diarias WHERE fecha_negocio >= DATE '2026-09-08' GROUP BY 1 ORDER BY 2")
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot(dia["fecha_negocio"], dia["ingreso"], color="#14304F"); a.set_title("Ingreso por día (MXN)"); a.tick_params(axis="x", rotation=45)
b.barh(ciudad["ciudad"], ciudad["ingreso"], color="#7AB929"); b.set_title("Ingreso por ciudad del cliente el día de la venta")
plt.tight_layout(); plt.show()

¿Ves la caída del último día? No es una caída: la carga es de madrugada y el día **va empezando**.
El tablero del kit lo avisa con una franja amarilla.

## Retos

1. **Un mart nuevo:** crea `elt/models/marts/mart_cancelaciones.sql` (cancelaciones por día y canal) y córrelo con
   `!cd elt && dbt build --profiles-dir . --select mart_cancelaciones`.
2. **Una prueba unitaria:** en `_intermediate.yml`, agrega una que verifique que un pedido en pesos no cambia de total.
3. **El snapshot fuera de orden:** corre
   `!cd elt && dbt build --profiles-dir . -s stg_clientes_foto snap_clientes --vars '{fecha_foto: "2026-09-10"}'`
   y luego `!cd elt && dbt test --profiles-dir . -s assert_historia_en_orden`. ¿Qué hizo dbt? ¿Qué hacía `scd2` en la 2.2?

## Cierre

| Hoy viste | Principio |
|---|---|
| Bronze leído sin copiar | En ELT, la transformación vive donde viven los datos |
| `ref()` y `source()` | El orden y el linaje salen del código |
| Seeds | Los catálogos chicos también se versionan |
| Snapshot | La SCD tipo 2 declarativa, si corre en orden |
| ETL = ELT | Mismo resultado; se elige por quién lo mantiene y dónde están los datos |
| `dbt build` con pruebas | Si una prueba de error falla, lo que sigue no se construye |

**Próxima sesión (3.1):** orquestación con Airflow. `python elt/correr.py` y `python -m transformacion.pipeline` se vuelven tareas de un DAG.